# Notebook 2: Deskriptive Statistik und Verteilungsdiagnostik

**Statistik für Data Science** · Kurs 3,906 · Bachelor Computer Science · HS2026
Universität St. Gallen · Begleitmaterial zur Übung

---

Dieses Notebook führt die Inhalte von **VL02 (Deskriptive Statistik und Verteilungsdiagnostik)** in Python aus. Jeder Unterabschnitt beginnt mit einem Problem, holt die passende Theorie aus der Vorlesung, löst das Problem im Code und sagt, was das Ergebnis bedeutet. Am Ende steht jeweils eine Frage für euren eigenen Datensatz.

**Die vier Schritte.** Sie kehren in jedem Unterabschnitt in derselben Reihenfolge wieder:

| Schritt | Was dort steht |
|---|---|
| **Problem.** | Eine konkrete Frage, meist mit einer Zahl aus den Daten |
| **Theorie.** | Was die Vorlesung dazu sagt, mit der Folie: *Vorlesung: Folie «…»* |
| **Code.** | `# Rechnen` berechnet die Statistik, `# Darstellen` zeigt vorher und nachher |
| **Bedeutung.** | Was der Output zeigt und welche Entscheidung daraus folgt |
| **Euer Datensatz.** | Dieselbe Frage für eure eigene Variable |

**Übersicht.** In der Übung zeigen wir die drei mit «im Raum» markierten Unterabschnitte. Danach übertragt ihr sie auf eure eigenen Daten (Abschnitt 4, «Transfer in der Übung»).

| | Unterabschnitt | Problem | Übung |
|---|---|---|---|
| **1** | **Lagekennzahlen** | **Wo liegt das Zentrum?** | |
| 1.1 | Mittelwert oder Median? | Zahlt ein typischer Passagier den Durchschnittspreis? | |
| 1.2 | Fehlende Werte | Mit welchem n rechnet pandas? | im Raum |
| 1.3 | Getrimmtes Mittel und Winsorisieren | Gibt es etwas zwischen Mittelwert und Median? | |
| 1.4 | Modus, Quantile und ECDF | Welche Fragen beantwortet kein Mittelwert? | |
| **2** | **Streuungskennzahlen** | **Wie stark streuen die Werte?** | |
| 2.1 | Varianz und Standardabweichung | Gleicher Mittelwert, gleich zuverlässig? | |
| 2.2 | Durch n oder n − 1 teilen? | Welche SD gehört in den Bericht? | im Raum |
| 2.3 | Verschieben und Skalieren | Was ändert eine Umrechnung der Einheit? | |
| 2.4 | IQR und Boxplot | Sind 116 markierte Ticketpreise 116 Fehler? | |
| 2.5 | MAD | Wie stark verändert ein Tippfehler die Streuung? | |
| **3** | **Verteilungsform** | **Welche Form hat die Verteilung?** | |
| 3.1 | Histogramm lesen | Reicht der Abstand von Mittelwert und Median als Diagnose? | |
| 3.2 | Modalität | Welchen Pinguin beschreibt die mittlere Flossenlänge? | im Raum |
| 3.3 | Heavy Tail oder isolierter Extremwert? | Sind 25 markierte Latenzen 25 Fehler? | |
| 3.4 | Kennzahlen allein reichen nicht | Zeigt `describe()` die Form? | |

**Datensätze.** Der Titanic-Datensatz führt die Analyse aus NB01 fort. Die Pinguindaten aus `seaborn` zeigen, was eine Umrechnung der Einheit bewirkt und warum mehrere Modi oft auf gemischte Gruppen hinweisen. Kleine simulierte Beispiele machen Eigenschaften sichtbar, die in realen Daten nicht kontrolliert verändert werden können.

## 0. Setup

Es gelten dieselben Konventionen wie in NB01: wenige Standardpakete und ein zentraler Seed. Beide Datensätze werden hier einmal geladen. Für die robusten Kennzahlen brauchen wir kein zusätzliches Paket; die wenigen Schritte werden bewusst ausgeschrieben.

In [ ]:
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

SEED = 42
RNG = np.random.default_rng(SEED)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (7.5, 4.2)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

In [ ]:
titanic = sns.load_dataset('titanic')
penguins = sns.load_dataset('penguins')

print(f'Titanic:  {titanic.shape[0]} Zeilen, {titanic.shape[1]} Spalten')
print(f'Pinguine: {penguins.shape[0]} Zeilen, {penguins.shape[1]} Spalten')

---

## 1. Lagekennzahlen

*Vorlesung: Abschnitt «1. Lagekennzahlen». Leitfrage: Was ist typisch?*

Ein Zentrum ist keine rein technische Wahl. Der Mittelwert berücksichtigt jeden Wert, der Median nur die Rangposition. Deshalb kann dieselbe Variable zwei sehr verschiedene Antworten auf die Frage nach dem typischen Wert geben.

### 1.1 Mittelwert oder Median?

**Problem.** Der durchschnittliche Ticketpreis auf der Titanic liegt bei 32.20. Zahlte ein typischer Passagier also rund 32? Tatsächlich zahlten 76.3 Prozent der Passagiere weniger als diesen Durchschnitt.

**Theorie.** *Vorlesung: Folien «Warum Lagekennzahlen wichtig sind», «Mittelwert: Definition & Eigenschaften», «Median: robustes Zentrum»*

Der Mittelwert $\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i$ berücksichtigt jeden Wert mit seiner Grösse. Der Median ist der mittlere Wert der sortierten Daten; für ihn zählt nur die Rangposition. Deshalb verschiebt ein einzelner extremer Wert den Mittelwert stark und den Median kaum.

**Code.** Zuerst das Einkommensbeispiel der Vorlesung mit und ohne den Wert 500, dann die Ticketpreise. Die Grafik zeigt das Einkommensbeispiel vorher und nachher.

In [ ]:
# Rechnen
einkommen = pd.Series([30, 32, 35, 40, 45, 50, 500], name='Einkommen')
ohne_500 = einkommen.iloc[:-1]

print(f'Mittelwert: {einkommen.mean():6.2f}   ohne 500: {ohne_500.mean():6.2f}')
print(f'Median:     {einkommen.median():6.2f}   ohne 500: {ohne_500.median():6.2f}')

In [ ]:
# Rechnen
fare = titanic['fare'].dropna()
unter_mittelwert = (fare < fare.mean()).mean()

print(f"n = {len(fare)}, fehlend = {titanic['fare'].isna().sum()}")
print(f'Mittelwert {fare.mean():.2f}, Median {fare.median():.2f}')
print(f'Minimum {fare.min():.2f}, Maximum {fare.max():.2f}')
print(f'Anteil der Passagiere unter dem Mittelwert: {unter_mittelwert:.1%}')

In [ ]:
# Darstellen
fig, achsen = plt.subplots(2, 1, figsize=(9, 3.8), sharex=True)
for ax, s, titel, farbe in zip(
    achsen,
    [ohne_500, einkommen],
    ['Vorher: ohne den Wert 500', 'Nachher: mit dem Wert 500'],
    ['#6B7280', '#7E22CE'],
):
    ax.scatter(s, np.zeros(len(s)), s=60, color=farbe, zorder=3)
    ax.axvline(s.mean(), color='#2166AC', linewidth=2, label=f'Mittelwert {s.mean():.2f}')
    ax.axvline(s.median(), color='#B45309', linewidth=2, linestyle='--', label=f'Median {s.median():.2f}')
    ax.set(title=titel, yticks=[])
    ax.legend(loc='upper right')
achsen[1].set_xlabel('Einkommen')
plt.tight_layout()
plt.show()

**Bedeutung.** Der einzelne Wert 500 verschiebt den Mittelwert von 38.67 auf 104.57, den Median nur von 37.5 auf 40.

Bei den Ticketpreisen liegt der Mittelwert mit 32.20 mehr als doppelt so hoch wie der Median 14.45, und 76.3 Prozent zahlten weniger. Für den **typischen** Ticketpreis trägt deshalb der Median. Der Mittelwert bleibt richtig, wenn die **Summe** zählt: Umsatz, gesamte Rechenzeit oder Gesamtkosten müssen den grossen Wert enthalten. Es gibt keine beste Lagekennzahl ohne fachliche Frage.

> **Euer Datensatz.** Welche metrische Variable beschreibt ihr zuerst? Fragt ihr nach dem typischen Wert oder nach einer Summe, und berichtet ihr deshalb den Median oder den Mittelwert?

### 1.2 Fehlende Werte: Mit welchem n rechnet pandas?

**Problem.** `titanic['age'].mean()` liefert 29.70 Jahre, ohne jede Warnung. Die Tabelle hat 891 Zeilen. Aus wie vielen Passagieren ist dieser Mittelwert berechnet, und welche fehlen?

**Theorie.** *Vorlesung: Folien «Mittelwert: Definition & Eigenschaften», «Python Basics Lage»; fehlende Werte aus VL01*

In der Formel $\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i$ ist n die Zahl der Werte, über die summiert wird.

Die Folie «Python Basics Lage» sagt: fehlende Werte (NaN) vor der Berechnung prüfen, dann entfernen oder ersetzen. Aus VL01: Ob Entfernen harmlos ist, hängt vom Mechanismus ab:

Bei MCAR (völlig zufällig fehlend) verzerrt das Weglassen den Mittelwert nicht, bei MAR und MNAR kann es ihn verzerren. Fehlen die Werte in einer Gruppe deutlich häufiger als in einer anderen, ist MCAR unplausibel.

**Code.** Erst zählen, dann dieselbe Zählung je Klasse: `size` zählt alle Zeilen, `count` nur die vorhandenen Werte. Die Grafik stellt beide je Klasse nebeneinander: alle Passagiere (vorher) und die, die tatsächlich in den Mittelwert eingehen (nachher).

In [ ]:
# Rechnen
alter = titanic['age']

print(f'Zeilen: {len(alter)}, vorhanden: {alter.count()}, fehlend: {alter.isna().sum()}')
print(f'Mittelwert: {alter.mean():.2f} Jahre, berechnet aus n = {alter.count()} Werten')

In [ ]:
# Rechnen
nach_klasse = titanic.groupby('class', observed=True)['age'].agg(['size', 'count', 'mean'])
nach_klasse['Anteil fehlend'] = 1 - nach_klasse['count'] / nach_klasse['size']
nach_klasse.round(3)

In [ ]:
# Darstellen
ax = nach_klasse[['size', 'count']].plot.bar(color=['#6B7280', '#7E22CE'], rot=0, figsize=(8.5, 4.2))
for container in ax.containers:
    ax.bar_label(container, fontsize=9)
ax.set(title='Wie viele Passagiere gehen in den Mittelwert des Alters ein?', xlabel='Klasse',
       ylabel='Anzahl')
ax.legend(['Vorher: alle Zeilen (size)', 'Nachher: mit Altersangabe (count)'], loc='upper left')
plt.show()

**Bedeutung.** `.mean()` überspringt fehlende Werte ohne Warnung: n ist 714, nicht 891. Die Tabelle zeigt denselben Unterschied je Klasse als Abstand zwischen `size` und `count`. Die Lücken sind nicht gleich verteilt: In der dritten Klasse fehlt das Alter bei 27.7 Prozent, in der zweiten bei 6.0 Prozent. MCAR ist damit unplausibel.

Das hat Folgen für den Mittelwert. Die dritte Klasse ist im Mittel die jüngste (25.1 Jahre) und verliert die meisten Werte, 136 von 491. Der Gesamtmittelwert von 29.70 Jahren kann deshalb zu hoch liegen; wie stark, hängt davon ab, warum die Werte fehlen. Deshalb gehören n und die Zahl der fehlenden Werte in jeden Bericht, und wenn die Lücken ungleich verteilt sind, auch ein Satz dazu.

> **Euer Datensatz.** Wie viele Werte fehlen in eurer Variable, mit welchem n rechnet pandas also? Fehlen sie in einer Gruppe häufiger als in einer anderen, und was bedeutet das für euren Mittelwert?

### 1.3 Getrimmtes Mittel und Winsorisieren (Ersetzen oder Löschen)

**Problem.** Der Median ignoriert, wie gross die Werte sind; der Mittelwert folgt jedem Extremwert. Bei den Ticketpreisen liegen beide mit 14.45 und 32.20 weit auseinander. Gibt es einen Mittelweg, der die Logik des Mittelwerts behält, aber die Ränder dämpft?

**Theorie.** *Vorlesung: Folien «Getrimmter Mittelwert», «Winsorisieren vs. Trimmen»*

Das getrimmte Mittel entfernt an beiden Rändern denselben Anteil und mittelt den Rest. Bei 10 Prozent Trimmen sind das **je 10 Prozent pro Rand**, insgesamt also ungefähr 20 Prozent. Winsorisieren entfernt nichts, sondern kappt die Randwerte auf die Grenzwerte. Beim Trimmen wird n kleiner, beim Winsorisieren bleibt n gleich. Die Vorlesung nutzt dafür `scipy.stats.trim_mean`; hier sind die Schritte ohne Zusatzpaket ausgeschrieben.

**Code.** Grenzwerte bestimmen, trimmen, winsorisieren, die vier Lagekennzahlen nebeneinander. Die Grafik zeigt die Ticketpreise vorher, getrimmt und winsorisiert.

In [ ]:
# Rechnen
anteil = 0.10
k = int(anteil * len(fare))                      # Zahl der Werte pro Rand
sortiert = np.sort(fare.to_numpy())

getrimmte_werte = sortiert[k:len(sortiert) - k]
winsorisierte_werte = fare.clip(lower=sortiert[k], upper=sortiert[-k - 1])
getrimmt = getrimmte_werte.mean()
winsorisiert = winsorisierte_werte.mean()
print(f'k = {k} Werte pro Rand, Grenzwerte {sortiert[k]:.2f} und {sortiert[-k - 1]:.2f}')

In [ ]:
# Darstellen
pd.DataFrame(
    {
        'Wert': [fare.mean(), fare.median(), getrimmt, winsorisiert],
        'n in der Berechnung': [len(fare), len(fare), len(getrimmte_werte), len(winsorisierte_werte)],
    },
    index=['Mittelwert', 'Median', '10%-getrimmtes Mittel', '10%-winsorisiertes Mittel'],
).round(2)

In [ ]:
# Darstellen
klassen = np.linspace(0, fare.max(), 61)

fig, achsen = plt.subplots(3, 1, figsize=(9, 7), sharex=True, sharey=True)
for ax, werte, titel, farbe in zip(
    achsen,
    [fare, getrimmte_werte, winsorisierte_werte],
    [f'Vorher: alle {len(fare)} Werte',
     f'Nachher, getrimmt: {len(getrimmte_werte)} Werte, Ränder entfernt',
     f'Nachher, winsorisiert: {len(winsorisierte_werte)} Werte, Ränder gekappt'],
    ['#6B7280', '#7E22CE', '#7E22CE'],
):
    ax.hist(werte, bins=klassen, color=farbe, edgecolor='white')
    ax.axvline(np.mean(werte), color='#2166AC', linewidth=2, label=f'Mittelwert {np.mean(werte):.2f}')
    ax.set(title=titel, ylabel='Anzahl')
    ax.legend(loc='upper right')
achsen[-1].set_xlabel('Ticketpreis')
plt.tight_layout()
plt.show()

**Bedeutung.** Beide robusten Mittel liegen zwischen Median (14.45) und Mittelwert (32.20): getrimmt 21.38, winsorisiert 25.65. Die Grafik zeigt, warum sie sich unterscheiden:

- **Trimmen** schneidet die lange Flanke ab; n sinkt von 891 auf 713.
- **Winsorisieren** behält alle 891 Werte, staut die hohen Preise aber beim Grenzwert 77.96.

Die Wahl muss dokumentiert werden. Starkes Trimmen ist bei kleinen Stichproben oder fachlich wichtigen Extremen keine neutrale Bereinigung: Es wirft Information weg.

> **Euer Datensatz.** Liegen Mittelwert und Median eurer Variable weit auseinander? Würdet ihr trimmen oder winsorisieren?

### 1.4 Modus, Quantile und ECDF (Empirical cumulative distribution function)

**Problem.** Drei Fragen, die kein Mittelwert beantwortet:

-  Welche Passagierklasse ist am häufigsten?
- Bis zu welchem Preis reichen 90 Prozent der Tickets?
- Welcher Anteil der Tickets kostet höchstens 50?

**Theorie.** *Vorlesung: Folien «Modus & Multimodalität», «Quantile & Perzentile», «ECDF und Quantile: zwei Leserichtungen»*

Der **Modus** ist der häufigste Wert. Für Kategorien ist das eindeutig interpretierbar; bei stetigen Messungen hängt er von Rundung und Messauflösung ab.

Ein **Quantil** $Q(p)$ ist der Grenzwert, den ein vorgegebener Anteil p der Daten nicht überschreitet: Q(0.25) ist das erste Quartil, Q(0.5) der Median, Q(0.9) eine operative Schwelle.

Die **ECDF** $\hat F(x)=\frac{1}{n}\sum_{i=1}^{n}\mathbb{1}(x_i\le x)$ liest dieselbe Beziehung rückwärts: zu einem Wert x der Anteil der Beobachtungen, die höchstens x sind. Die Folie definiert das Quantil als direkte Umkehrung der ECDF, $Q(p)=\min\{x:\hat F(x)\ge p\}$.

**Code.** Modus, Quantile und die ECDF in beiden Leserichtungen, dann dasselbe Quantil mit zwei Methoden. Die Grafik zeigt an der ECDF genau die beiden Fragen aus dem Problem: vom Preis 50 zum Anteil und vom Anteil 90 Prozent zum Preis.

In [ ]:
# Rechnen
print('Modus der Passagierklasse:', titanic['class'].mode().tolist())
print('\nHäufigste Klassen:')
print(titanic['class'].value_counts())

In [ ]:
# Rechnen
quantile = fare.quantile([0.25, 0.50, 0.75, 0.90, 0.95])
quantile.index = ['Q1 (25 %)', 'Median (50 %)', 'Q3 (75 %)', 'P90', 'P95']
print(quantile.round(2))

# ECDF an der Stelle 50: Zahl der Preise bis 50, geteilt durch n
schwelle = 50
anzahl_bis_50 = (fare <= schwelle).sum()
anteil_bis_50 = anzahl_bis_50 / len(fare)
print(f'\nPreis -> Anteil (ECDF): {anzahl_bis_50} von {len(fare)} Ticketpreisen sind höchstens {schwelle}, '
      f'also {anteil_bis_50:.1%}.')
print(f'Anteil -> Preis (Quantil): 90 % der Ticketpreise sind höchstens {quantile["P90"]:.2f}.')

In [ ]:
# Rechnen
# Dasselbe Quantil, zwei Methoden: pandas interpoliert, die Folie kehrt die ECDF direkt um.
klein = pd.Series([2, 4, 6, 8, 50])
print(f"Fünf Werte, P90:  pandas {klein.quantile(0.9):.2f}   "
      f"Umkehrung der ECDF {np.quantile(klein, 0.9, method='inverted_cdf'):.2f}")
print(f"Ticketpreise, P90: pandas {fare.quantile(0.9):.2f}   "
      f"Umkehrung der ECDF {np.quantile(fare, 0.9, method='inverted_cdf'):.2f}")

In [ ]:
# Darstellen
x = np.sort(fare.to_numpy())
y = np.arange(1, len(x) + 1) / len(x)
p90 = quantile['P90']

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.step(x, y, where='post', color='#7E22CE', linewidth=1.8)

# Preis -> Anteil: von 50 senkrecht zur Treppe, dann waagrecht zur y-Achse
ax.plot([schwelle, schwelle], [0, anteil_bis_50], color='#15803D', linestyle='--', linewidth=1.5,
        label=f'Preis {schwelle} → Anteil {anteil_bis_50:.1%}')
ax.annotate('', xy=(0, anteil_bis_50), xytext=(schwelle, anteil_bis_50),
            arrowprops=dict(arrowstyle='->', color='#15803D', linewidth=1.8))

# Anteil -> Preis: von 0.9 waagrecht zur Treppe, dann senkrecht zur x-Achse
ax.plot([0, p90], [0.9, 0.9], color='#B45309', linestyle='--', linewidth=1.5,
        label=f'Anteil 90 % → Preis {p90:.2f}')
ax.annotate('', xy=(p90, 0), xytext=(p90, 0.9),
            arrowprops=dict(arrowstyle='->', color='#B45309', linewidth=1.8))

ax.set(title='ECDF der Ticketpreise: zwei Leserichtungen', ylabel='Anteil ≤ x',
       xlabel=f'Ticketpreis (Achse bei 150 abgeschnitten, {(fare > 150).sum()} Preise liegen darüber)',
       xlim=(0, 150), ylim=(0, 1.02))
ax.legend(loc='lower right')
plt.show()

**Bedeutung.** Jede Frage hat ihr eigenes Werkzeug. Der Modus der Klasse ist «Third». Das Quantil beantwortet die Schwellenfrage: 90 Prozent der Ticketpreise liegen höchstens bei 77.96. Die ECDF beantwortet die Anteilsfrage: 82.0 Prozent kosten höchstens 50. In der Grafik steigt die Treppe bei jeder Beobachtung um 1/n. Grün: vom Preis 50 senkrecht zur Treppe, dann waagrecht zur y-Achse, das ergibt 82.0 Prozent (731 von 891). Orange: vom Anteil 0.9 waagrecht zur Treppe, dann senkrecht zur x-Achse, das ergibt 77.96. Dieselbe Kurve beantwortet also beide Fragen, nur in entgegengesetzter Richtung.

Bei den fünf Werten liefern die beiden Methoden 33.20 und 50.00, bei den 891 Ticketpreisen fallen sie zusammen. Bei kleinen Datensätzen gehören Quantilmethode und Paketversion deshalb zur Dokumentation.

> **Euer Datensatz.** Gibt es in eurem Projekt eine Schwelle, die zählt, etwa eine Servicegrenze oder einen Grenzwert? Welches Quantil beschreibt sie, und wie gross ist euer n?

> **Take-Home Lage**
>
> 1. **Mittelwert** bei ungefähr symmetrischen Daten oder wenn die Summe zählt.
> 2. **Median** bei Schiefe und Extremwerten.
> 3. **Getrimmtes Mittel** als dokumentierter Kompromiss, wenn nur die Ränder stören.
> 4. **Quantile** für Schwellen, **Modus** primär für Kategorien.
> 5. **n und fehlende Werte** gehören zu jeder Lagekennzahl.
>
> Ein Zentrum ohne Verteilungsform ist unvollständig.

---

## 2. Streuungskennzahlen

*Vorlesung: Abschnitt «2. Streuungskennzahlen». Leitfrage: Wie weit liegen die Werte auseinander?*

Ein Zentrum allein genügt nicht. Zwei Gruppen können denselben Mittelwert haben und völlig verschieden streuen. Streuung braucht immer eine Bezugsgrösse und eine Einheit.

### 2.1 Varianz und Standardabweichung

**Problem.** Zwei Buslinien, je drei gemessene Wartezeiten. Linie A: 2, 4 und 6 Minuten. Linie B: 0, 4 und 8 Minuten. Beide haben im Mittel 4 Minuten Wartezeit. Sind sie gleich zuverlässig, und wie drückt man den Unterschied in einer Zahl aus?

**Theorie.** *Vorlesung: Folien «Varianz und Standardabweichung», «Varianz: die mittlere Fläche der Abweichungsquadrate»*

Die Varianz mittelt die quadrierten Abweichungen vom Mittelwert: $v=\frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})^2$. Jede Abweichung wird zu einem Quadrat; die Varianz ist die mittlere Fläche und steht in quadrierter Einheit (min²). Die Standardabweichung $\mathrm{SD}=\sqrt{v}$ holt die Seitenlänge zurück in die Originaleinheit (min). Beide sind empfindlich gegenüber Ausreissern.

**Code.** Für beide Linien die quadrierten Abweichungen, die Varianz und die SD. Die Grafik stellt beide Linien nebeneinander und zeichnet jede Abweichung als Quadrat.

In [ ]:
# Rechnen
linie_a = pd.Series([2, 4, 6])
linie_b = pd.Series([0, 4, 8])

# Linie A: Abweichung vom Mittelwert, quadrieren, mitteln (durch n), Wurzel
abweichung_a = linie_a - linie_a.mean()
varianz_a = (abweichung_a ** 2).mean()
sd_a = np.sqrt(varianz_a)

# Linie B: dieselben vier Schritte
abweichung_b = linie_b - linie_b.mean()
varianz_b = (abweichung_b ** 2).mean()
sd_b = np.sqrt(varianz_b)

print(f'Linie A: Abweichungen {abweichung_a.tolist()}, Varianz {varianz_a:.2f} min², SD {sd_a:.2f} min')
print(f'Linie B: Abweichungen {abweichung_b.tolist()}, Varianz {varianz_b:.2f} min², SD {sd_b:.2f} min')

In [ ]:
# Darstellen
mittel = 4   # beide Linien haben den Mittelwert 4

fig, achsen = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, linie, titel in [
    (achsen[0], linie_a, f'Linie A: Varianz {varianz_a:.2f} min², SD {sd_a:.2f} min'),
    (achsen[1], linie_b, f'Linie B: Varianz {varianz_b:.2f} min², SD {sd_b:.2f} min'),
]:
    for wert in linie:
        seite = abs(wert - mittel)            # Abstand zum Mittelwert = Seitenlänge des Quadrats
        links = min(wert, mittel)
        ax.add_patch(plt.Rectangle((links, 0), seite, seite, facecolor='#7E22CE', alpha=0.25,
                                   edgecolor='#7E22CE', linewidth=1.5))
        if seite > 0:
            ax.text(links + seite / 2, seite / 2, f'{seite}² = {seite ** 2} min²', ha='center')
    ax.scatter(linie, [0, 0, 0], s=60, color='#7E22CE', zorder=3)
    ax.axvline(mittel, color='#2166AC', linewidth=2, label='Mittelwert 4 min')
    ax.set(title=titel, xlabel='Wartezeit in Minuten', ylabel='Abweichung in Minuten',
           xlim=(-0.5, 8.5), ylim=(-0.3, 4.5), aspect='equal')
    ax.grid(False)
achsen[0].legend(loc='upper right')
plt.tight_layout()
plt.show()

**Bedeutung.** Beide Linien haben denselben Mittelwert von 4 Minuten, aber bei Linie B sind die Quadrate viel grösser: Varianz 10.67 statt 2.67 min², SD 3.27 statt 1.63 Minuten.

Die Varianz in Quadratminuten ist schwer zu lesen. Die SD steht wieder in Minuten und lässt sich berichten:

Linie B streut doppelt so stark wie Linie A. Ein Mittelwert ohne Streuung verschweigt, wie verlässlich er ist.

> **Euer Datensatz.** Welche Einheit hat eure Variable, und welche Einheit haben damit Varianz und SD? Welche der beiden Zahlen würdet ihr berichten?

### 2.2 Durch n oder n − 1 teilen?

**Problem.** Dieselben drei Wartezeiten der Linie A, zwei Bibliotheken: pandas meldet eine Standardabweichung von 2.00 Minuten, NumPy 1.63 Minuten. Keine der beiden rechnet falsch. Welche Zahl gehört in den Bericht?

**Theorie.** *Vorlesung: Folien «Varianz: Durch n oder n−1 teilen?», «Achtung: unterschiedliche Standards»*

Die Folie stellt zwei Nenner nebeneinander. Mit n beschreibt man die Streuung genau der vorliegenden Werte. Mit n − 1 schätzt man aus einer Stichprobe die Varianz der Population: Der Mittelwert wird aus denselben Daten berechnet und passt sich an sie an, deshalb unterschätzt der Nenner n die Populationsvarianz im Durchschnitt. In pandas und NumPy heisst die Wahl `ddof`: `ddof=0` teilt durch n, `ddof=1` durch n − 1.

Die Korrektur macht eine schlecht gezogene Stichprobe **nicht repräsentativ**. Sie gleicht nur die Unterschätzung durch den geschätzten Mittelwert aus.

**Code.** Beide Nenner auf Linie A, die Voreinstellungen von pandas und NumPy, und wie stark sich die beiden bei wachsendem n unterscheiden. Dann eine Simulation, die die Theorie prüft: 10 000 Stichproben mit je drei Werten aus einer Population mit bekannter Varianz 1. Die Grafik zeigt die geschätzten Varianzen vorher (durch n) und nachher (durch n − 1).

In [ ]:
# Rechnen
print(f'Varianz  ddof=0: {linie_a.var(ddof=0):.2f}   ddof=1: {linie_a.var(ddof=1):.2f}')
print(f'SD       ddof=0: {linie_a.std(ddof=0):.2f}   ddof=1: {linie_a.std(ddof=1):.2f}')
print()
print(f'pandas  linie_a.std()              ohne Angabe: {linie_a.std():.2f}')
print(f'NumPy   np.std(linie_a.to_numpy()) ohne Angabe: {np.std(linie_a.to_numpy()):.2f}')

In [ ]:
# Rechnen
# 10 000 Stichproben mit je n = 3 aus einer Normalverteilung mit bekannter Varianz 1
rng_sim = np.random.default_rng(SEED)
stichproben = rng_sim.normal(loc=0, scale=1, size=(10_000, 3))
var_ddof0 = stichproben.var(axis=1, ddof=0)
var_ddof1 = stichproben.var(axis=1, ddof=1)

print('Wahre Varianz der Population:            1.000')
print(f'Durchschnitt der Schätzungen mit ddof=0: {var_ddof0.mean():.3f}')
print(f'Durchschnitt der Schätzungen mit ddof=1: {var_ddof1.mean():.3f}')

In [ ]:
# Darstellen
klassen = np.linspace(0, 6, 61)

fig, achsen = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, werte, titel, farbe in zip(
    achsen,
    [var_ddof0, var_ddof1],
    ['Vorher: durch n geteilt (ddof=0)', 'Nachher: durch n − 1 geteilt (ddof=1)'],
    ['#6B7280', '#7E22CE'],
):
    ax.hist(werte, bins=klassen, color=farbe, edgecolor='white')
    ax.axvline(werte.mean(), color='#2166AC', linewidth=2, label=f'Durchschnitt {werte.mean():.3f}')
    ax.axvline(1, color='black', linestyle='--', linewidth=1.5, label='wahre Varianz 1')
    ax.set(title=titel, xlabel='geschätzte Varianz', ylabel='Anzahl Stichproben')
    ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

**Bedeutung.** Die Formel auf der Folie teilt durch n, pandas teilt ohne Angabe durch n − 1, NumPy durch n. Dieselben drei Wartezeiten ergeben so 1.63 oder 2.00 Minuten, je nach Bibliothek. Die Simulation bestätigt die Theorie: Durch n geteilt liegen die Schätzungen im Durchschnitt bei 0.666, also rund ein Drittel zu tief; durch n − 1 geteilt bei 0.998, nahe an der wahren Varianz 1.

Der Unterschied schrumpft mit n. Bei n = 3 ist die SD mit `ddof=1` um 22.5 Prozent grösser, bei n = 30 um 1.7 Prozent, bei den 891 Ticketpreisen um 0.1 Prozent. Die Entscheidung bleibt trotzdem nötig: Wer nur die vorliegenden Daten beschreibt, teilt durch n (`ddof=0`); wer auf eine Population schliesst, durch n − 1 (`ddof=1`). Deshalb `ddof` immer ausdrücklich setzen und im Bericht nennen.

### 2.3 Verschieben und Skalieren

**Problem.** Die Flossenlängen der Pinguine sind in Millimetern erfasst. Für einen Bericht rechnet ihr sie in Zentimeter um, oder ein Messgerät addiert einen festen Versatz. Was passiert mit Mittelwert, Varianz und SD, und ändert sich die Form der Verteilung?

**Theorie.** *Vorlesung: Folie «Eigenschaften von Varianz und SD»*

Für veränderte Daten $y_i=a\,x_i+b$ gilt: Mittelwert $\bar{y}=a\,\bar{x}+b$, Varianz $v_y=a^2\,v_x$, Standardabweichung $\mathrm{SD}_y=|a|\,\mathrm{SD}_x$. Verschieben um b lässt die Abstände zum Mittelwert gleich; Skalieren mit a multipliziert sie mit |a|.

**Code.** Mittelwert, Varianz und SD für drei Varianten derselben Werte. Die Grafik zeigt die Flossenlängen in mm (vorher), in cm und um 100 verschoben (nachher).

In [ ]:
# Rechnen
flipper_mm = penguins['flipper_length_mm'].dropna()
flipper_cm = flipper_mm / 10            # skalieren mit a = 0.1
flipper_plus_100 = flipper_mm + 100     # verschieben um b = 100

skalierung = pd.DataFrame(
    {
        'Mittelwert': [flipper_mm.mean(), flipper_cm.mean(), flipper_plus_100.mean()],
        'Varianz (ddof=1)': [flipper_mm.var(ddof=1), flipper_cm.var(ddof=1), flipper_plus_100.var(ddof=1)],
        'SD (ddof=1)': [flipper_mm.std(ddof=1), flipper_cm.std(ddof=1), flipper_plus_100.std(ddof=1)],
    },
    index=['mm', 'cm (geteilt durch 10)', 'mm plus 100'],
)
skalierung.round(2)

In [ ]:
# Darstellen
fig, achsen = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
sns.histplot(flipper_mm, bins=20, color='#6B7280', ax=achsen[0])
achsen[0].set(title='Vorher: in mm', xlabel='Flossenlänge (mm)', ylabel='Anzahl')
sns.histplot(flipper_cm, bins=20, color='#7E22CE', ax=achsen[1])
achsen[1].set(title='Nachher: in cm', xlabel='Flossenlänge (cm)')
sns.histplot(flipper_plus_100, bins=20, color='#7E22CE', ax=achsen[2])
achsen[2].set(title='Nachher: in mm, plus 100', xlabel='Flossenlänge (mm) + 100')
plt.tight_layout()
plt.show()

**Bedeutung.** Die drei Histogramme haben exakt dieselbe Form; nur die Achse ändert sich. Geteilt durch 10 wird die SD von 14.06 mm zu 1.41 cm, die Varianz von 197.73 mm² zu 1.98 cm², also durch 100 geteilt. Das Verschieben um 100 ändert nur den Mittelwert, nicht die Streuung. Deshalb Streuung immer mit Einheit berichten, und zwar als SD: Sie trägt die Einheit der Daten, die Varianz deren Quadrat.

> **Euer Datensatz.** Rechnet ihr eine Variable um, etwa Einheiten, Währungen oder Zeitzonen? Welche Kennzahlen ändern sich dabei, und in welcher Einheit berichtet ihr?

### 2.4 IQR und Boxplot

**Problem.** Die 1.5·IQR-Regel markiert 116 von 891 Ticketpreisen als auffällig, 13.0 Prozent. Sind das 116 Datenfehler, die man löschen sollte?

**Theorie.** *Vorlesung: Folien «IQR als robuste Streuung», «Anatomie des Boxplots»*

Der Interquartilsabstand `IQR = Q3 − Q1` ist die Breite der mittleren 50 Prozent. Er wird mit dem Median berichtet und reagiert kaum auf wenige extreme Randwerte. Ein Boxplot codiert genau diese Grössen:

- Box von Q1 bis Q3, ihre Breite ist der IQR. Linie in der Box: Median.
- Unterer und oberer Zaun: `Q1 − 1.5·IQR` und `Q3 + 1.5·IQR`.
- Whisker: äusserste **beobachtete** Werte innerhalb der Zäune, nicht Minimum und Maximum.
- Punkte jenseits der Whisker: Kandidaten für eine Prüfung, kein Urteil.

Die Zäune gehen auf John Tukey zurück, der den Boxplot eingeführt hat, und heissen deshalb **Tukey-Zäune** (englisch *Tukey Fences*; Tukey 1977, *Exploratory Data Analysis*, auf der Folie «Kernressourcen»). Werte ausserhalb der Zäune zu markieren nennt man die **1.5·IQR-Regel**. VL03 vergleicht sie mit anderen Regeln für Ausreisser.

**Code.** Erst die Zäune über alle Ticketpreise, dann dieselbe Regel innerhalb jeder Klasse. Die Grafik zeigt den Boxplot über alle Passagiere (vorher) und getrennt nach Klasse (nachher).

In [ ]:
# Rechnen
q1, median, q3 = fare.quantile([0.25, 0.50, 0.75])
iqr = q3 - q1
unterer_zaun = q1 - 1.5 * iqr
oberer_zaun = q3 + 1.5 * iqr
markiert_global = (fare < unterer_zaun) | (fare > oberer_zaun)

print(f'Q1 = {q1:.2f}, Median = {median:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}')
print(f'Tukey-Zäune: {unterer_zaun:.2f} bis {oberer_zaun:.2f}')
print(f'Markiert: {markiert_global.sum()} von {len(fare)} ({markiert_global.sum() / len(fare):.1%})')

In [ ]:
# Rechnen
# Dieselbe Regel wie oben, aber jede Klasse bekommt ihre eigenen Quartile und Zäune
for klasse in ['First', 'Second', 'Third']:
    preise = fare[titanic['class'] == klasse]
    q1_k, q3_k = preise.quantile([0.25, 0.75])
    iqr_k = q3_k - q1_k
    unterer_zaun_k = q1_k - 1.5 * iqr_k
    oberer_zaun_k = q3_k + 1.5 * iqr_k
    markiert_k = (preise < unterer_zaun_k) | (preise > oberer_zaun_k)
    print(f'{klasse:6s}  Zäune {unterer_zaun_k:7.2f} bis {oberer_zaun_k:6.2f}   '
          f'markiert: {markiert_k.sum()} von {len(preise)}')

In [ ]:
# Darstellen
fig, achsen = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
sns.boxplot(y=fare, color='#6B7280', ax=achsen[0])
achsen[0].set(title='Vorher: alle Klassen zusammen', ylabel='Ticketpreis')
sns.boxplot(data=titanic, x='class', y='fare', color='#7E22CE', ax=achsen[1])
achsen[1].set(title='Nachher: je Klasse', xlabel='Klasse', ylabel='')
plt.tight_layout()
plt.show()

**Bedeutung.** Über alle Passagiere markiert die Regel 116 Preise, innerhalb der Klassen zusammen 79 (20 + 7 + 52), und zum Teil andere. Der obere Zaun der dritten Klasse liegt bei 27.12, der globale bei 65.63: Ein Ticketpreis um 30 ist in der dritten Klasse auffällig, über alle Passagiere gesehen nicht. Ob ein Wert auffällig ist, hängt also davon ab, womit man ihn vergleicht.

Das ist die wichtige Grenze der Regel: **1.5·IQR ist ein Screening-Werkzeug, kein Löschbefehl.** Ob ein Wert Messfehler, seltener legitimer Fall oder Teil einer langen Flanke ist, entscheidet erst die fachliche Prüfung. Der systematische Umgang mit Ausreissern folgt in VL03.

> **Euer Datensatz.** Wie viele Werte markiert die 1.5·IQR-Regel in eurer Variable, einmal über alle Daten und einmal je Gruppe? Welche davon prüft ihr, und womit?

### 2.5 MAD: robuste Abstände zum Median

**Problem.** Acht Messwerte, ein Tippfehler: 100 statt 55. Nur ein Wert ist falsch, und n bleibt gleich. Wie stark ändert sich die gemessene Streuung, je nachdem, mit welcher Kennzahl man sie misst?

**Theorie.** *Vorlesung: Folien «MAD robuste Alternative», «Klassisch vs. robust im Vergleich», «Gute Berichtssprache»*

Die median absolute deviation $\mathrm{MAD}=\mathrm{median}(|x-\tilde{x}|)$ ist der typische absolute Abstand zum Median $\tilde{x}$. Für einen Vergleich mit der Standardabweichung unter Normalverteilung wird sie oft mit 1.4826 skaliert; Roh-MAD und skalierte MAD sind nicht dieselbe Zahl. Die Vorlesung stellt klassische (SD) und robuste Masse (IQR, MAD) gegenüber und berichtet sie paarweise: Mittelwert mit SD, Median mit IQR oder MAD.

**Code.** Die Streuungsmasse vor und nach dem Tippfehler, dann ein Berichtssatz für den Ticketpreis nach dem Muster der Vorlesung. Die erste Grafik stellt die Masse vor und nach dem Tippfehler nebeneinander. Die zweite zeigt an den echten Ticketpreisen, was die beiden Paare aus dem Berichtssatz beschreiben: Mittelwert ± SD (vorher) und Median mit IQR (nachher).

In [ ]:
# Rechnen
vorher = pd.Series([44, 47, 49, 50, 50, 52, 53, 55])
nachher = vorher.replace(55, 100)       # der Tippfehler

# MAD: Median der absoluten Abstände zum Median
mad_vorher = (vorher - vorher.median()).abs().median()
mad_nachher = (nachher - nachher.median()).abs().median()

streuung = pd.DataFrame(
    {
        'Vorher': [
            vorher.std(ddof=1),
            vorher.quantile(0.75) - vorher.quantile(0.25),
            mad_vorher,
            1.4826 * mad_vorher,
        ],
        'Nachher: 55 durch 100 ersetzt': [
            nachher.std(ddof=1),
            nachher.quantile(0.75) - nachher.quantile(0.25),
            mad_nachher,
            1.4826 * mad_nachher,
        ],
    },
    index=['SD (ddof=1)', 'IQR', 'MAD', 'MAD skaliert'],
)
streuung.round(2)

In [ ]:
# Darstellen
ax = streuung.plot.bar(color=['#6B7280', '#7E22CE'], rot=0, figsize=(8.5, 4.2))
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=9)
ax.set(title='Ein Tippfehler: 100 statt 55', ylabel='Streuung')
plt.show()

In [ ]:
# Rechnen
q1, q3 = fare.quantile([0.25, 0.75])
median = fare.median()
iqr = q3 - q1

print(
    f'Robust: Der Ticketpreis liegt typisch bei {median:.2f} (Median); '
    f'die mittleren 50 % liegen zwischen {q1:.2f} und {q3:.2f} (IQR {iqr:.2f}). '
    f'n = {len(fare)}, fehlend = {titanic["fare"].isna().sum()}.'
)
print(
    f'Klassisch ergänzend: Mittelwert {fare.mean():.2f} , '
    f'SD {fare.std(ddof=1):.2f}  (ddof=1).'
)

In [ ]:
# Darstellen
mittelwert = fare.mean()
sd = fare.std(ddof=1)
klassen = np.arange(0, 155, 5)

fig, achsen = plt.subplots(2, 1, figsize=(9, 6), sharex=True, sharey=True)

# Vorher: klassisches Paar, Mittelwert ± SD
achsen[0].hist(fare, bins=klassen, color='#6B7280', edgecolor='white')
achsen[0].axvspan(mittelwert - sd, mittelwert + sd, color='#2166AC', alpha=0.15,
                  label=f'Mittelwert ± SD: {mittelwert - sd:.2f} bis {mittelwert + sd:.2f}')
achsen[0].axvline(mittelwert, color='#2166AC', linewidth=2, label=f'Mittelwert {mittelwert:.2f}')
achsen[0].set(title='Vorher: klassisch berichtet, Mittelwert ± SD', ylabel='Anzahl')
achsen[0].legend(loc='upper right')

# Nachher: robustes Paar, Median und IQR
achsen[1].hist(fare, bins=klassen, color='#7E22CE', edgecolor='white')
achsen[1].axvspan(q1, q3, color='#B45309', alpha=0.2, label=f'IQR: {q1:.2f} bis {q3:.2f}')
achsen[1].axvline(median, color='#B45309', linewidth=2, linestyle='--', label=f'Median {median:.2f}')
achsen[1].set(title='Nachher: robust berichtet, Median und IQR', ylabel='Anzahl', xlim=(-25, 150),
              xlabel=f'Ticketpreis (Achse bei 150 abgeschnitten, {(fare > 150).sum()} Preise liegen darüber)')
achsen[1].legend(loc='upper right')
plt.tight_layout()
plt.show()

**Bedeutung.** Die SD springt beim oberen Beispiel von 3.46 auf 18.15, auf mehr als das Fünffache. IQR und MAD bleiben in diesem Beispiel unverändert. Das ist der praktische Unterschied zwischen klassischen und robusten Massen. Daraus folgen die Paare:

- ungefähr symmetrisch: **Mittelwert und SD**;
- schief, heavy-tailed oder mit relevanten Extremwerten: **Median und IQR**, optional MAD;
- nicht mischen: `Median ± SD` verbindet zwei Kennzahlen mit unterschiedlichem Zentrum und unterschiedlicher Robustheit.

Die zweite Grafik zeigt, warum beim Ticketpreis das robuste Paar in den Bericht gehört. Mittelwert ± SD reicht von −17.49 bis 81.90, also bis unter 0, obwohl es keine negativen Preise gibt, und der Mittelwert liegt rechts von den meisten Werten. Median und IQR beschreiben dagegen, wo die Preise tatsächlich liegen: Die mittlere Hälfte kostet zwischen 7.91 und 31.00.

Der Berichtssatz oben folgt der Folie «Gute Berichtssprache»: Lage, passende Streuung, Einheit, n und fehlende Werte.

> **Euer Datensatz.** Schreibt den Satz für eure Variable: Lage und passende Streuung, Einheit, n, fehlende Werte und das gewählte `ddof`.

> **Take-Home Streuung**
>
> 1. **Varianz und SD** sind empfindlich; die SD steht in der Originaleinheit.
> 2. **IQR** beschreibt die mittleren 50 Prozent robust.
> 3. **MAD** nutzt typische Abstände zum Median; Skalierung angeben.
> 4. **`ddof` explizit setzen**, Einheit, `n` und fehlende Werte nennen.
>
> Jede Streuungszahl braucht eine Bezugsgrösse und eine Einheit.

---

## 3. Verteilungsform

*Vorlesung: Abschnitt «3. Verteilungsform». Leitfrage: Welche Form hat die Verteilung, und tragen die gewählten Kennzahlen?*

Lage und Streuung sind Kompressionen. Erst ein Bild zeigt Modi, Lücken und lange Flanken. Deshalb gilt die Reihenfolge: **Form ansehen, dann Kennzahlen wählen, dann berichten.**

### 3.1 Histogramm lesen

**Problem.** Beim Ticketpreis liegt der Mittelwert mit 32.20 mehr als doppelt so hoch wie der Median 14.45. Das spricht für eine lange rechte Flanke. Reicht dieser Abstand als Diagnose der Form, oder kann er täuschen?

**Theorie.** *Vorlesung: Folien «Histogramm: Form sichtbar machen», «Schiefe: wenn Mittelwert und Median auseinanderlaufen»*

Ein Histogramm zählt Werte in Intervallen; die Höhe ist die Häufigkeit. Die Vorlesung unterscheidet vier Formen: symmetrisch, rechtsschief, linksschief und multimodal. Die Schiefe wird nach der langen Flanke benannt, nicht nach dem Ort des Modus. Die Differenz von Mittelwert und Median ist ein schneller Formindikator. Die Klassenzahl ist eine Darstellungsentscheidung; ihre systematische Wahl folgt in VL03.

**Code.** Zuerst das Histogramm der Ticketpreise. Dann vier simulierte Formen: vorher nur Mittelwert und Median als Zahlen, nachher als Histogramm.

In [ ]:
# Darstellen
fig, ax = plt.subplots(figsize=(8.5, 4.4))
sns.histplot(fare, bins=30, color='#7E22CE', edgecolor='white', ax=ax)
ax.axvline(fare.mean(), color='#2166AC', linewidth=2, label=f'Mittelwert {fare.mean():.2f}')
ax.axvline(fare.median(), color='#B45309', linewidth=2, linestyle='--', label=f'Median {fare.median():.2f}')
ax.set(title='Verteilung der Titanic-Ticketpreise', xlabel='Ticketpreis', ylabel='Anzahl')
ax.legend()
plt.show()

In [ ]:
# Rechnen
rng_form = np.random.default_rng(SEED)
form_daten = {
    'symmetrisch': rng_form.normal(50, 8, 800),
    'rechtsschief': 35 + rng_form.lognormal(2.6, 0.65, 800),
    'linksschief': 100 - rng_form.lognormal(2.6, 0.65, 800),
    'bimodal': np.r_[rng_form.normal(35, 4, 400), rng_form.normal(65, 5, 400)],
}

for name, werte in form_daten.items():
    print(f'{name:13s} Mittelwert {np.mean(werte):6.2f}   Median {np.median(werte):6.2f}   '
          f'Differenz {np.mean(werte) - np.median(werte):5.2f}')

In [ ]:
# Darstellen
fig, achsen = plt.subplots(2, 2, figsize=(11, 7))
for ax, (name, werte) in zip(achsen.flat, form_daten.items()):
    sns.histplot(werte, bins=28, color='#7E22CE', edgecolor='white', ax=ax)
    ax.axvline(np.mean(werte), color='#2166AC', linewidth=2, label='Mittelwert')
    ax.axvline(np.median(werte), color='#B45309', linewidth=2, linestyle='--', label='Median')
    ax.set(title=name, xlabel='Wert', ylabel='Anzahl')
achsen[0, 0].legend()
plt.tight_layout()
plt.show()

**Bedeutung.** Die Ticketpreise sind **rechtsschief**: Die meisten Werte liegen links, die dünne Flanke reicht bis über 500, und der Mittelwert liegt rechts vom Median. Die vier simulierten Formen zeigen die Grenze der Kennzahl: Im bimodalen Beispiel liegen Mittelwert (50.19) und Median (48.42) beide im Tal zwischen den beiden Modi, wo kaum Werte liegen; ihr Abstand ist sogar kleiner als im rechtsschiefen Beispiel. Ein kleiner Abstand beweist also weder Symmetrie noch Unimodalität. Jede Form führt zu einer anderen Beschreibung:

| Form | Erkennungsmerkmal | Sinnvolle Beschreibung |
|---|---|---|
| symmetrisch | beide Flanken ähnlich | Mittelwert und SD |
| rechtsschief | lange rechte Flanke, oft Mittelwert > Median | Median, IQR und obere Quantile |
| linksschief | lange linke Flanke, oft Mittelwert < Median | Median, IQR und untere Quantile |
| multimodal | mehrere Modi mit Tälern dazwischen | Gruppen suchen und getrennt berichten |

> **Euer Datensatz.** Zeichnet ein Histogramm eurer Variable. Welche der vier Formen seht ihr, und welche Kennzahlen folgen daraus?

### 3.2 Modalität: eine Population oder eine Mischung?

**Problem.** Die mittlere Flossenlänge aller Pinguine im Datensatz beträgt 200.92 mm. Welchen Pinguin beschreibt dieser Wert?

**Theorie.** *Vorlesung: Folie «Modalität: ein Gipfel oder mehrere?»*

Eine Verteilung mit zwei Modi heisst bimodal, mit mehreren multimodal. Die Folie sagt: Mehrere Modi deuten oft auf gemischte Gruppen hin, und eine gemeinsame Kennzahl beschreibt dann keine der Gruppen. Wenn Gruppen erkennbar sind, wird pro Gruppe berichtet. Der Plot zeigt eine Struktur, aber noch keine Ursache.

**Code.** Lage und Modus über alle Pinguine, dann die Flossenlängen aller Arten zusammen (vorher) und nach Art getrennt (nachher), zuletzt Lage und Streuung je Art mit `groupby`.

In [ ]:
# Rechnen
flipper = penguins['flipper_length_mm'].dropna()

print(f"Fehlende Flossenlängen: {penguins['flipper_length_mm'].isna().sum()}")
print(f'Alle Pinguine: Mittelwert {flipper.mean():.2f} mm, Median {flipper.median():.1f} mm')
print('Modus der Flossenlänge:', flipper.mode().tolist())

In [ ]:
# Darstellen
fig, achsen = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True, sharey=True)
sns.histplot(flipper, bins=18, color='#6B7280', edgecolor='white', ax=achsen[0])
achsen[0].axvline(flipper.mean(), color='#2166AC', linewidth=2,
                  label=f'Mittelwert aller {flipper.mean():.2f} mm')
achsen[0].set(title='Vorher: alle Arten zusammen', xlabel='Flossenlänge (mm)', ylabel='Anzahl')
achsen[0].legend(loc='upper right')

sns.histplot(
    data=penguins, x='flipper_length_mm', hue='species', bins=18,
    element='step', fill=False, common_norm=False, ax=achsen[1]
)
achsen[1].set(title='Nachher: nach Art getrennt', xlabel='Flossenlänge (mm)', ylabel='Anzahl')
plt.tight_layout()
plt.show()

In [ ]:
# Rechnen
penguins.groupby('species')['flipper_length_mm'].agg(['count', 'mean', 'median', 'std']).round(1)

**Bedeutung.** Links ist die Verteilung bimodal. Rechts wird sichtbar, woher das kommt: Adelie und Chinstrap überlappen im unteren Modus, Gentoo bildet den oberen. Der Gesamtmittelwert von 200.92 mm liegt zwischen den Modi und beschreibt keine Art gut: Adelie 190.0, Chinstrap 195.8, Gentoo 217.2 mm. Auch der Modus hilft wenig. Der häufigste Einzelwert, 190 mm, sagt nichts über die Gentoo-Pinguine.

Pro Gruppe berichten ist im Code eine Zeile: `groupby` teilt nach einer beobachteten Variable, `agg` rechnet die Kennzahlen je Gruppe, und `count` zeigt das n jeder Gruppe ohne fehlende Werte. Nach welcher Variable ihr gruppiert, folgt aber nicht aus dem Plot. Das ist eine fachliche Entscheidung, die ihr begründen müsst.

> **Euer Datensatz.** Hat eure Variable mehr als einen Modus? Welche beobachtete Variable in eurem Datensatz könnte die Gruppen erklären? Berichtet Lage und Streuung pro Gruppe und begründet die Gruppierung in eurem Log.

### 3.3 Heavy Tail oder isolierter Extremwert?

**Problem.** In 500 simulierten Latenzmessungen markiert die 1.5·IQR-Regel am oberen Rand 25 Werte. In einer zweiten Messreihe mit 500 Bearbeitungszeiten markiert sie dort genau einen. Sind das 25 Fehler und ein Fehler?

**Theorie.** *Vorlesung: Folie «Heavy Tails: die lange Flanke»; Tukey-Zäune aus Abschnitt 2.4*

Die 1.5·IQR-Regel stammt aus Abschnitt 2.4: Auffällig ist, was über dem oberen Tukey-Zaun `Q3 + 1.5·IQR` oder unter dem unteren `Q1 − 1.5·IQR` liegt. Hier geht es um grosse Werte, deshalb zählt nur der obere Zaun.

Ein grosser Wert ist nicht allein deshalb ein Fehler. Bei Latenzen, Einkommen oder Schadenshöhen kann eine lange Flanke zur Verteilung gehören: Ein Heavy Tail dünnt sich schrittweise aus. Ein isolierter Kandidat steht dagegen oft nach einer sichtbaren Lücke. Die Folie fasst es so: Die Form entscheidet, ob ein grosser Wert überhaupt auffällig ist.

**Code.** Zwei simulierte Messreihen mit bekannter Wahrheit: oberer Zaun, Zahl der markierten Werte und grösste Lücke zwischen zwei sortierten Werten. Die Grafik zeigt beide als Punktstreifen mit dem oberen Zaun.

In [ ]:
# Rechnen
rng_tail = np.random.default_rng(7)
heavy_tail = pd.Series(rng_tail.lognormal(mean=2.7, sigma=0.65, size=500), name='Latenz: Heavy Tail')
# Der Hauptkörper wird begrenzt, damit genau der angehängte Wert isoliert steht.
hauptkoerper = np.clip(rng_tail.normal(20, 3, 499), 12, 27)
isoliert = pd.Series(np.r_[hauptkoerper, 78], name='Bearbeitungszeit: isolierter Wert')

zaeune = {}
for s in [heavy_tail, isoliert]:
    q1, q3 = s.quantile([0.25, 0.75])
    zaeune[s.name] = q3 + 1.5 * (q3 - q1)
    ueber_zaun = (s > zaeune[s.name]).sum()
    luecke = s.sort_values().diff().max()
    print(f'{s.name:34s} oberer Zaun {zaeune[s.name]:5.2f}   Werte über dem Zaun: {ueber_zaun:2d}   '
          f'grösste Lücke {luecke:5.2f}')

In [ ]:
# Darstellen
fig, achsen = plt.subplots(2, 1, figsize=(10, 4.4))
for ax, s, farbe in zip(achsen, [heavy_tail, isoliert], ['#7E22CE', '#B45309']):
    sns.stripplot(x=s, orient='h', jitter=0.18, size=2.5, alpha=0.5, color=farbe, ax=ax)
    ax.axvline(zaeune[s.name], color='black', linestyle='--', label='oberer Tukey-Zaun')
    ax.set(title=s.name, xlabel='Wert', ylabel='')
    ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

**Bedeutung.** In der Latenzreihe markiert die Regel 25 Werte, aber im Punktstreifen dünnen sie sich schrittweise aus, ohne klaren Bruch. Das ist kein Beweis für 25 Datenfehler; es zeigt, dass die Standardregel nicht zur Form passt. In der zweiten Reihe trennt eine Lücke von 51.00 den einzelnen Wert vom Rest, in der ersten ist die grösste Lücke 12.85. Auch der isolierte Punkt ist nur ein Kandidat. Die Entscheidung braucht Dokumentation und Domänenwissen.

Wichtig: **Heavy-tailed bedeutet nicht automatisch rechtsschief.** Es gibt auch symmetrische Verteilungen mit dicken Flanken. Diese feinere Diagnose und der QQ-Plot folgen in VL03.

> **Euer Datensatz.** Hat eure Variable eine lange Flanke oder einzelne Werte nach einer Lücke? Wie viele Werte markiert die Regel, und passt sie zur Form eurer Verteilung?

### 3.4 Kennzahlen allein reichen nicht

**Problem.** `describe()` fasst eine Variable in acht Zahlen zusammen. Kann man daran ablesen, dass die Flossenlänge bimodal ist?

**Theorie.** *Vorlesung: Folie «Deskriptive Statistik: drei Fragen an jede Variable»*

Die Vorlesung stellt an jede Variable drei Fragen: Wo liegt das Zentrum, wie stark streuen die Werte, welche Form hat die Verteilung? Der Merksatz dazu: Erst die Form ansehen, dann Lage und Streuung wählen, dann berichten. `describe()` beantwortet nur die ersten beiden Fragen. In ihm stecken zwei Entscheidungen von oben: `count` ist das n ohne fehlende Werte, und `std` rechnet mit `ddof=1`.

**Code.** Die `describe()`-Tabelle für Alter, Ticketpreis und Flossenlänge (vorher), dann die drei Histogramme mit Mittelwert und Median (nachher).

In [ ]:
# Rechnen
pd.DataFrame(
    {
        'Alter (Jahre)': titanic['age'].describe(),
        'Ticketpreis': titanic['fare'].describe(),
        'Flossenlänge (mm)': penguins['flipper_length_mm'].describe(),
    }
).round(2)

In [ ]:
# Darstellen
variablen = {
    'Alter (Jahre)': titanic['age'].dropna(),
    'Ticketpreis': fare,
    'Flossenlänge (mm)': flipper,
}

fig, achsen = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (name, werte) in zip(achsen, variablen.items()):
    sns.histplot(werte, bins=30, color='#7E22CE', edgecolor='white', ax=ax)
    ax.axvline(werte.mean(), color='#2166AC', linewidth=2, label='Mittelwert')
    ax.axvline(werte.median(), color='#B45309', linewidth=2, linestyle='--', label='Median')
    ax.set(title=name, xlabel=name, ylabel='Anzahl')
achsen[0].legend()
fig.suptitle('Nachher: dieselben drei Variablen als Histogramm')
plt.tight_layout()
plt.show()

**Bedeutung.** Die Tabelle bündelt Lage und Streuung, ersetzt aber keine Grafik:

- Beim **Alter** liegen Mittelwert und Median (`50%`) nahe zusammen, und `count` zeigt 714 statt 891 Werte.
- Beim **Ticketpreis** liegen Mittelwert und Median weit auseinander. Das passt zur sichtbaren Rechtsschiefe.
- Bei der **Flossenlänge** liegen Mittelwert und Median mit 200.92 und 197.00 nahe zusammen. Trotzdem ist die Verteilung bimodal, weil Arten gemischt sind. Genau daran scheitert jede Diagnose, die nur auf Kennzahlen beruht.

Die drei Fragen der Vorlesung lassen sich nur mit Kennzahlen und Bild zusammen beantworten.

> **Euer Datensatz.** Beantwortet die drei Fragen für eure Variable: Lage, Streuung, Form. Was davon liefert `describe()`, und wofür braucht ihr das Histogramm?

---

## 4. Zusammenfassung und Übertragung

Die drei Lernziele der Vorlesung, jeweils mit der Stelle, an der sie im Notebook laufen:

| # | Lernziel | Abschnitt |
|---|---|---|
| 01 | **Lage** passend zur Form bestimmen | 1 |
| 02 | **Streuung** klassisch und robust berechnen und berichten | 2 |
| 03 | **Verteilungsform** über Symmetrie, Schiefe, Modalität und Tails lesen | 3 |

Die wichtigsten drei Sätze des Notebooks:

1. **Erst die Form, dann die Kennzahl.** Ein Zentrum ohne Histogramm kann mitten in einem Tal liegen.
2. **Klassisch gehört zu klassisch, robust zu robust.** Mittelwert mit SD; Median mit IQR oder MAD.
3. **Ein markierter Wert ist ein Prüfkandidat, kein Löschauftrag.** Die Verteilungsform und das Domänenwissen entscheiden.

### Was ihr mit eurem eigenen Datensatz macht

Die Fragen **Euer Datensatz** in den Abschnitten 1 bis 3 führen durch diese Schritte.

**1. Eine metrische Variable und ihre Einheit festlegen.** Prüft das Messniveau aus NB01. Nennt Einheit, Zahl der vorhandenen Werte und Zahl der fehlenden Werte.

**2. Die Form ansehen.** Zeichnet ein Histogramm. Sucht nach Symmetrie, einer langen linken oder rechten Flanke, mehreren Modi und Lücken. Die Klassenzahl ist nicht neutral; variiert sie zur Sensitivitätsprüfung, ohne ein gefälliges Bild auszuwählen.

**3. Lage passend wählen.** Bei ungefähr symmetrischer, unimodaler Form Mittelwert; bei Schiefe Median; bei fachlich begründeter Randstörung eventuell ein dokumentiertes getrimmtes Mittel. Für Servicegrenzen zusätzlich P90 oder P95.

**4. Die passende Streuung danebenstellen.** SD zum Mittelwert, IQR und optional MAD zum Median. `ddof` und bei MAD die Skalierung explizit angeben.

**5. Mehrere Modi untersuchen.** Sucht eine beobachtete Gruppierungsvariable und berichtet pro fachlich sinnvoller Gruppe. Ein Gesamtmittel über eine Mischung kann niemanden repräsentieren.

**6. Tukey-Punkte prüfen, nicht löschen.** Kontrolliert Quelle, Einheit, Erfassungsprozess und fachliche Plausibilität. Vergleicht Kennzahlen mit und ohne Kandidaten höchstens als Sensitivitätsanalyse und dokumentiert jede Entscheidung.

**7. Einen vollständigen Satz schreiben.** Nennt Kennzahl, Streuung, Einheit, `n`, fehlende Werte und die gewählte Konvention. Zahlen ohne Datengrundlage sind kein Bericht.

**Was VL02 bewusst noch nicht beantwortet.** Die systematische Wahl der Histogramm-Klassenbreite, KDE, QQ-Plots und formale Ausreisserdiagnostik gehören zu VL03. Zusammenhänge zwischen zwei Variablen folgen in VL04. Eine deskriptive Diagnose beschreibt die beobachteten Daten; sie beweist weder eine Ursache noch die Übertragbarkeit auf eine Population.

---

**Weiter in VL03:** Diagnoseplots und Ausreisser, also wie empfindlich Histogramme sind, was ein QQ-Plot über Tails zeigt und wie auffällige Beobachtungen fachlich geprüft werden.